In [ ]:
!pip install openai koreanize-matplotlib

In [ ]:
# ============================================================
# 실습 2-3. 논문 초록 구조화 추출
# 대상: Attention Is All You Need (Vaswani et al., 2017)
# 목표: Temperature 0.1로 사실 기반 추출, T 비교로 품질 차이 체감
# 사전 설치: pip install openai koreanize-matplotlib
# ============================================================

import os
import tiktoken
import koreanize_matplotlib

# ── API 키 설정 ──────────────────────────────────────────────
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Colab Secrets에서 API 키 로드 완료")
except Exception:
    pass
# os.environ["OPENAI_API_KEY"] = "sk-..."

if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError(
        "API 키가 설정되지 않았습니다.\n"
        "방법 A: 왼쪽 사이드바 열쇠 아이콘 -> 'OPENAI_API_KEY' 등록\n"
        "방법 B: os.environ['OPENAI_API_KEY'] = 'sk-...' 주석 해제"
    )

from openai import OpenAI
client = OpenAI()
print(f"API 키 확인: sk-...{os.environ['OPENAI_API_KEY'][-4:]}")


# ============================================================
# [실습 변수] 여기만 수정하여 다양한 실험 가능
# ============================================================

# 구조화 추출 대상 논문 초록 (한글 번역본)
# 본인 관심 논문의 초록으로 교체 가능
ABSTRACT = """
지배적인 시퀀스 변환 모델은 인코더와 디코더를 포함하는 복잡한 순환 신경망
또는 합성곱 신경망을 기반으로 한다. 가장 성능이 우수한 모델들은 어텐션
메커니즘을 통해 인코더와 디코더를 연결한다. 본 논문에서는 순환 구조와
합성곱 구조를 완전히 배제하고 오직 어텐션 메커니즘에만 기반한 새로운
단순한 네트워크 아키텍처인 트랜스포머를 제안한다. 두 기계 번역 과제에서
수행한 실험 결과, 트랜스포머 모델은 기존 최고 성능 모델들보다 우수하면서도
병렬화가 훨씬 용이하고 학습 시간도 크게 단축되었다. 영어-독일어 번역
과제에서 본 모델은 앙상블 모델을 포함한 기존 최고 성능 모델보다 2 BLEU
이상 높은 28.4 BLEU를 달성하였다. 영어-프랑스어 번역 과제에서는 단일
모델 기준 최고 성능인 41.0 BLEU를 달성하였으며, 이는 기존 최고 성능
모델들의 학습 비용 중 일부에 불과한 비용으로 학습된 결과이다. 트랜스포머는
영어 구문 분석 과제에서도 대규모 및 제한적 학습 데이터 모두에서 성공적으로
일반화되어 우수한 성능을 보였다.
"""
PAPER_TITLE  = "Attention Is All You Need (Vaswani et al., 2017)"
PAPER_URL    = "https://arxiv.org/abs/1706.03762"

# 구조화 추출 온도: 사실 기반 작업 -> 낮게 설정
EXTRACT_TEMP = 0.1

# Temperature 비교 실험 값 목록
# 0.1(사실 기반) vs 0.8(다소 해석 포함) 차이 비교
COMPARE_TEMPS = [0.1, 0.8]

# 섹션별 요약 대상 (본인 보고서 섹션으로 교체 가능)
# {섹션명: 섹션 내용} 형태
REPORT_SECTIONS = {
    "배경":    "기존 RNN 및 LSTM 기반 시퀀스 모델은 순차 처리 특성으로 인해 병렬화가 어렵고, 장거리 의존성 학습에 한계가 있었다.",
    "모델 구조": "트랜스포머는 인코더-디코더 구조를 유지하되, 순환 구조 대신 멀티헤드 셀프어텐션과 포지션별 피드포워드 네트워크로 구성된다.",
    "실험":    "WMT 2014 영어-독일어 번역에서 28.4 BLEU, 영어-프랑스어에서 41.0 BLEU를 달성하여 기존 최고 성능을 능가하였다.",
    "결론":    "어텐션 메커니즘만으로 구성된 트랜스포머는 기계 번역 외 다양한 시퀀스 태스크에 적용 가능하며 병렬 학습이 가능하다.",
}

# ============================================================
# 이하 코드는 수정 불필요
# ============================================================

# ── 공통 함수 ─────────────────────────────────────────────────
def chat(system_prompt, user_prompt,
         temperature=0.1, max_tokens=600):
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return resp.choices[0].message.content, resp.usage

def section(title):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")


# ── 블록 1: 논문 정보 및 토큰 확인 ──────────────────────────
enc = tiktoken.get_encoding("cl100k_base")
abstract_tokens = len(enc.encode(ABSTRACT))

print(f"논문: {PAPER_TITLE}")
print(f"출처: {PAPER_URL}")
print(f"초록 길이: {abstract_tokens} 토큰")
print("-> 1일차에서 배운 Transformer 구조가 이 논문에서 처음 제안됨")


# ── 블록 2: 구조화 추출 (Temperature 0.1) ────────────────────
section("블록 2. 논문 초록 구조화 추출")

# 4요소(역할·지시·형식·조건) 적용
# 실습 2-2에서 배운 프롬프트 설계 원칙을 그대로 적용
sys_extract = """역할: 당신은 AI 및 자연어처리 분야 논문 분석 전문가입니다.
지시: 주어진 논문 초록을 아래 형식으로 정확하게 구조화하세요.
형식:
[연구 목적] ...
[연구 방법] ...
[주요 결과] ...
[한계점]   ...
[핵심 키워드] 쉼표로 구분, 5개 이내
조건: 원문에 없는 내용은 추가하지 마세요. 각 항목은 2문장 이내로 작성하세요."""

ans_struct, usage = chat(
    system_prompt=sys_extract,
    user_prompt=f"다음 초록을 분석하세요:\n{ABSTRACT}",
    temperature=EXTRACT_TEMP,
)
print(f"\n[구조화 추출 결과]  Temperature={EXTRACT_TEMP}")
print(ans_struct)
print(f"\n입력 토큰: {usage.prompt_tokens} / 출력 토큰: {usage.completion_tokens}")
print("-> Temperature 0.1: 원문 사실만 추출, 추론 최소화")


# ── 블록 3: Temperature 비교 ─────────────────────────────────
section("블록 3. Temperature 비교 (요약 품질 차이)")

compare_results = {}
for t in COMPARE_TEMPS:
    ans_t, _ = chat(
        system_prompt=sys_extract,
        user_prompt=f"다음 초록을 분석하세요:\n{ABSTRACT}",
        temperature=t,
        max_tokens=400,
    )
    compare_results[t] = ans_t
    print(f"\n[Temperature={t}]")
    print(ans_t[:500])
    print()

print("-> 요약/추출 작업은 Temperature 0.1~0.3 권장 - 낮을수록 원문 사실만 추출")
print("-> T=0.8에서는 '특히', '주요' 같은 강조 표현이 추가되거나 해석이 포함될 수 있음")


# ── 블록 4: 섹션별 분할 요약 ─────────────────────────────────
section("블록 4. 섹션별 분할 요약 (긴 보고서 처리 전략)")

# 핵심: 긴 문서를 한 번에 넣지 않고 섹션별로 분리하여 각각 요약
# 이유: 토큰 한도 절약 + 각 섹션 요약 품질 유지
sys_section = """역할: 당신은 연구 보고서 편집자입니다.
지시: 각 섹션을 핵심 1문장으로 요약하세요.
조건: 원문에 없는 내용을 추가하지 마세요."""

summaries = {}
print("\n섹션별 1문장 요약 결과:")
print("-" * 55)
for sec_name, content in REPORT_SECTIONS.items():
    ans_s, _ = chat(
        system_prompt=sys_section,
        user_prompt=f"[{sec_name}]\n{content}",
        temperature=0.1,
        max_tokens=100,
    )
    summaries[sec_name] = ans_s.strip()
    print(f"  [{sec_name}] {ans_s.strip()}")

print("\n-> 긴 문서는 섹션별로 분할 요약 후 통합 - 토큰 한도 초과 방지")


# ── 블록 5: 통합 요약 생성 ───────────────────────────────────
section("블록 5. 섹션별 요약 -> 통합 요약 생성")

# 섹션별 요약을 이어붙여 전체 요약을 만드는 2단계 요약 전략
combined = "\n".join(f"[{k}] {v}" for k, v in summaries.items())

sys_integrate = """역할: 당신은 논문 요약 전문가입니다.
지시: 섹션별 요약을 하나의 자연스러운 단락으로 통합하세요.
조건: 3~5문장으로 간결하게 작성하세요."""

ans_integrated, _ = chat(
    system_prompt=sys_integrate,
    user_prompt=f"다음 섹션별 요약을 통합하세요:\n{combined}",
    temperature=0.3,
    max_tokens=200,
)
print(f"\n통합 요약:\n{ans_integrated}")
print("\n-> 2단계 요약 전략: 섹션별 요약 -> 통합 요약")
print("   토큰을 절약하면서도 전체 맥락을 유지하는 방법")
print("-> 다음 실습: 이 논문의 연구 배경을 기반으로 연구 가설을 생성합니다")